# 01 — Kafka Consumer: Messages, Offsets and Consumer Groups

The provided stack has a `kafka-producer` container, which should already
be running (you can check with `docker compose ps`) and it is pushinig 20
messages every 5 seconds.

We will be exploring the consumption aspect in an interactive manner,
within this notebook.

In [ ]:
from kafka import KafkaConsumer
import json
import pandas as pd

BOOTSTRAP = "kafka:9092"
TOPIC     = "server-metrics-raw"

def deserialize(m):
    return json.loads(m.decode("utf-8"))

print("Imports OK")

## A naive consumer

An "anonymous" consumer (from the point of view of Kafka) is shown below:

The `consumer_timeout_ms` parameter tells the consumer to stop blocking
after 1 second with no new messages — otherwise it would wait forever.

In [ ]:
# Stateless/anonymous consumer
consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=BOOTSTRAP,
    auto_offset_reset="earliest",   # start from the beginning of the topic
    consumer_timeout_ms=1_000,      # stop after 1 s of inactivity
    value_deserializer=deserialize,
)

all_messages = [msg.value for msg in consumer]
consumer.close()

df = pd.DataFrame(all_messages)
print(f"Read {len(df):,} messages")
df.head()

You can see a message `Read #### messages` after running the cell above.

**Now run the consumer cell again** without restarting the kernel. The number of read messages should have changed.

### Exercise 1 `latest` vs `earliest`:

In the `auto_offset_reset` parameter before, try to use `"latest"` instead of `"earliest"`. You may want to run it multiple times, sometimes it will read 0 messages, sometimes it will read a certain amount.

Do you see the difference between both options?

In [ ]:
consumer = ...

## Consumer that tracks the position

A **consumer group** is identified by a `group_id` string.  When a consumer
with a group_id reads messages, Kafka stores that internally.
The next run of a consumer with the **same group_id** will start from that
stored offset instead of the beginning.

Setting `enable_auto_commit=True` tells the consumer library to commit the
offset automatically as messages are read.

Run the cell below **twice**.  Compare the number of messages returned.

In [ ]:
# Consumer with a group_id
GROUP_ID = "lab-group"

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=BOOTSTRAP,
    group_id=GROUP_ID,
    auto_offset_reset="earliest",   # only matters for a brand-new group, first execution
    enable_auto_commit=True,        # commit offset as we read
    consumer_timeout_ms=1_000,
    value_deserializer=deserialize,
)

batch = [msg.value for msg in consumer]
consumer.close()

print(f"Group '{GROUP_ID}' — read {len(batch):,} messages")
if batch:
    first = batch[0]
    last  = batch[-1]
    print(f"  First message timestamp : {first['timestamp']}")
    print(f"  Last  message timestamp : {last['timestamp']}")

### Exercise 2 Different groups

Create a new `KafkaConsumer` with a different `group_id`.

This new consumer should be independent of the previous one. Try to run it again and how many messages it is reading.

For each batch of a consumer run, can there be overlap in their first and last message timestamp?

Can there be overlap between batches of different consumers? Why?

In [ ]:
consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=BOOTSTRAP,
    group_id=...,  # EXERCISE: Fill this
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    consumer_timeout_ms=1_000,
    value_deserializer=deserialize,
)

...